# 🧾 SmartReceipt Analyzer — Klasifikasi Kategori Item Struk
**TF-IDF (character n-gram) + Naive Bayes** · Dataset: `Dataset_SmartReceipt_Sintetis.csv` (15 kategori)

Alur model (sesuai proposal PBL): `Teks item struk (hasil OCR) → Preprocessing → TF-IDF → Naive Bayes → Kategori`

**Kategori:** Makanan, Minuman, Rokok & Tembakau, Bayi & Anak, Perawatan Diri, Kesehatan, Rumah Tangga, Hewan Peliharaan,
Transportasi, Tagihan & Utilitas, Elektronik, Pakaian & Fashion, Pendidikan & ATK, Hiburan & Hobi, Lainnya.

**Isi notebook**
1. Load & eksplorasi dataset
2. Preprocessing + simulasi *noise* OCR
3. Tuning model pada data validasi
4. Evaluasi pada data uji (produk yang **belum pernah dilihat**)
5. Evaluasi lebih ketat (GroupKFold per merek) & kalibrasi ambang keyakinan
6. Training model final, fungsi prediksi, demo berbagai jenis struk
7. Export JSON untuk Flutter (on-device, tanpa API)
8. Uji dengan struk asli milik kamu (opsional, **sangat disarankan**)

> ⚠️ **Catatan jujur:** dataset ini **sintetis** (dibuat dari daftar produk + variasi penulisan khas struk), bukan hasil scan struk sungguhan.
> Angka akurasi di sini adalah perkiraan optimis. Angka yang sebenarnya baru terlihat setelah diuji dengan struk asli (bagian 8).

## 1. Import Library & Load Dataset

In [ ]:
import re, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

In [ ]:
# Lokal / VS Code : taruh CSV di folder yang sama dengan notebook ini.
# Google Colab    : jalankan sel ini, lalu upload file CSV saat diminta.
CSV_NAME = "Dataset_SmartReceipt_Sintetis.csv"

if not Path(CSV_NAME).exists():
    try:
        from google.colab import files
        print("Upload file", CSV_NAME)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Letakkan {CSV_NAME} di folder yang sama dengan notebook ini.")

df = pd.read_csv(CSV_NAME)
print("Jumlah baris :", len(df))
print("Produk unik  :", df.loc[df["sumber"] == "sintetis", "product_id"].nunique())
print("Kata kunci generik:", df.loc[df["sumber"] == "kata_kunci", "product_id"].nunique())
df.sample(10, random_state=SEED)[["receipt_text", "kategori", "gaya", "split"]]

## 2. Eksplorasi Data
- `receipt_text` = tulisan item seperti yang mungkin muncul di struk (singkatan, huruf kapital, terpotong, tanpa spasi, kode PLU)
- `kategori` = label
- `split` = **pembagian per produk** (train / val / test). Satu produk hanya ada di satu split, jadi data uji berisi produk yang
  benar-benar belum pernah dilihat model. Pembagian ini juga terstratifikasi per kategori.
- `grup` = kunci merek/jenis produk (dipakai untuk evaluasi yang lebih ketat di bagian 5)
- `sumber` = `sintetis` (produk bermerek/spesifik) atau `kata_kunci` (istilah generik per kategori, mis. "sabun cuci piring", "tiket pesawat").
  Kata kunci generik **hanya masuk ke data training**, supaya model paham kata umum walaupun mereknya belum dikenal.

In [ ]:
ringkas = df.pivot_table(index="kategori", columns="sumber", values="product_id", aggfunc="nunique", fill_value=0)
ringkas = ringkas.rename(columns={"sintetis": "produk", "kata_kunci": "kata_kunci"})
ringkas["baris"] = df.groupby("kategori").size()
ringkas = ringkas.sort_values("produk", ascending=False)
display(ringkas)

print("Distribusi split (jumlah produk+kata kunci / baris):")
display(df.groupby("split").agg(produk=("product_id", "nunique"), baris=("id", "count")))
display(pd.crosstab(df["kategori"], df["split"]))

fig, ax = plt.subplots(figsize=(8, 5))
ringkas["produk"].sort_values().plot.barh(ax=ax, color="#4C78A8")
ax.set_title("Jumlah produk unik per kategori"); plt.tight_layout(); plt.show()

In [ ]:
# Contoh variasi tulisan struk untuk beberapa produk
for nama in ["Indomie Goreng", "Pertamax", "Token Listrik PLN", "Ayam Geprek", "Panadol Biru"]:
    cuplikan = df[df["nama_produk"] == nama][["receipt_text", "gaya"]]
    print(f"--- {nama} ({df.loc[df.nama_produk == nama, 'kategori'].iloc[0]})")
    print(cuplikan.to_string(index=False, header=False))

## 3. Preprocessing & Simulasi Noise OCR

In [ ]:
def preprocess(teks: str) -> str:
    """huruf kecil -> hapus angka/ukuran (600ml, 1kg, kode PLU) -> hapus simbol -> rapikan spasi."""
    teks = str(teks).lower()
    teks = re.sub(r"\b\d+([.,]\d+)?\s*(ml|liter|l|kg|gr|gram|g|pcs)?\b", " ", teks)
    teks = re.sub(r"[^a-z\s]", " ", teks)
    return re.sub(r"\s+", " ", teks).strip()

for c in ["AQUA BTL 600ML", "8992761134019 INDOMIE GRG 85G", "TOKEN LISTRIK 50000", "2 PERTAMAX 5,2 L"]:
    print(f"{c!r:38} -> {preprocess(c)!r}")

In [ ]:
# Karakter yang sering tertukar saat OCR
CONFUSABLE = {
    "i": ["l", "1", "j"], "l": ["i", "1"], "o": ["0", "c"], "0": ["o"], "s": ["5"],
    "b": ["6", "h"], "a": ["e", "o"], "e": ["c", "a"], "u": ["v"], "v": ["u"],
    "m": ["n"], "n": ["m", "r"], "g": ["q", "9"], "q": ["g"],
}

def add_ocr_noise(teks: str, rng: random.Random, p: float = 0.08) -> str:
    """Tambahkan kesalahan OCR acak: tukar huruf mirip, hapus, gandakan, tukar posisi."""
    out, i = [], 0
    while i < len(teks):
        ch = teks[i]
        if ch != " " and rng.random() < p:
            aksi = rng.choice(["ganti", "ganti", "hapus", "gandakan", "tukar"])
            if aksi == "ganti":
                out.append(rng.choice(CONFUSABLE.get(ch.lower(), [ch])))
            elif aksi == "gandakan":
                out.extend([ch, ch])
            elif aksi == "tukar" and i + 1 < len(teks) and teks[i + 1] != " ":
                out.extend([teks[i + 1], ch]); i += 1
            elif aksi != "hapus":
                out.append(ch)
        else:
            out.append(ch)
        i += 1
    return "".join(out)

rng_demo = random.Random(1)
print([add_ocr_noise("INDOMIE GORENG", rng_demo, p=0.12) for _ in range(6)])

In [ ]:
def siapkan(frame):
    """DataFrame mentah -> (raw, label, text) setelah preprocessing."""
    out = pd.DataFrame({"raw": frame["receipt_text"].values, "label": frame["kategori"].values})
    out["text"] = out["raw"].map(preprocess)
    return out[out["text"].str.len() > 0].reset_index(drop=True)

def versi_noise(frame, seed, p_min=0.04, p_max=0.15):
    """Salinan frame dengan kolom receipt_text diberi noise OCR."""
    rng = random.Random(seed)
    out = frame.copy()
    out["receipt_text"] = [add_ocr_noise(t, rng, p=rng.uniform(p_min, p_max)) for t in frame["receipt_text"]]
    return out

def augment(frame, n_noisy, seed):
    """Data asli + n_noisy salinan ber-noise (dipakai HANYA untuk training)."""
    bagian = [siapkan(frame)] + [siapkan(versi_noise(frame, seed * 100 + i)) for i in range(n_noisy)]
    return pd.concat(bagian, ignore_index=True).drop_duplicates(["text", "label"]).reset_index(drop=True)

def buat_model(ngram=(2, 5), alpha=0.03, fit_prior=True, max_features=20000):
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=ngram, sublinear_tf=True,
                                  max_features=max_features)),
        ("nb", MultinomialNB(alpha=alpha, fit_prior=fit_prior)),
    ])

train = df[df["split"] == "train"]
val   = df[df["split"] == "val"]
test  = df[df["split"] == "test"]

train_aug = augment(train, n_noisy=6, seed=1)
print("Sampel training setelah augmentasi:", len(train_aug))

## 4. Tuning Model (pakai data **validasi**)
Data uji tidak disentuh dulu. Pilihan terbaik ditentukan dari **macro-F1** pada data validasi (teks bersih + teks ber-noise OCR),
supaya semua kategori diperhitungkan sama rata walaupun jumlah produknya berbeda.

In [ ]:
val_all = pd.concat([siapkan(val), siapkan(versi_noise(val, seed=7))], ignore_index=True)

hasil = []
for ng in [(2, 4), (2, 5), (3, 5)]:
    for alpha in [0.01, 0.03, 0.1, 0.3]:
        for fp in [True, False]:
            m = buat_model(ng, alpha, fp).fit(train_aug["text"], train_aug["label"])
            p = m.predict(val_all["text"])
            hasil.append({"ngram": ng, "alpha": alpha, "fit_prior": fp,
                          "akurasi": accuracy_score(val_all["label"], p),
                          "macro_f1": f1_score(val_all["label"], p, average="macro")})

tuning = pd.DataFrame(hasil).sort_values("macro_f1", ascending=False).reset_index(drop=True)
display(tuning.head(8).style.format({"akurasi": "{:.1%}", "macro_f1": "{:.3f}"}))

terbaik = tuning.iloc[0]
BEST = dict(ngram=tuple(terbaik["ngram"]), alpha=float(terbaik["alpha"]), fit_prior=bool(terbaik["fit_prior"]))
print("Konfigurasi terpilih:", BEST)

## 5. Evaluasi pada Data Uji
Produk di data uji **tidak ada** di data training. Ini mengukur kemampuan menebak kategori **produk baru**
(termasuk saat teksnya salah baca OCR).

In [ ]:
model_eval = buat_model(**BEST).fit(train_aug["text"], train_aug["label"])

test_bersih = siapkan(test)
test_noise  = siapkan(versi_noise(test, seed=8))

baris = []
for nama, dset in [("Teks bersih", test_bersih), ("Teks + noise OCR", test_noise)]:
    p = model_eval.predict(dset["text"])
    baris.append({"Skenario": nama, "Jumlah": len(dset),
                  "Akurasi": accuracy_score(dset["label"], p),
                  "Macro-F1": f1_score(dset["label"], p, average="macro")})
display(pd.DataFrame(baris).style.format({"Akurasi": "{:.1%}", "Macro-F1": "{:.3f}"}))

# Pembanding: produk yang SUDAH dikenal (ada di training) dengan noise baru
dikenal = siapkan(versi_noise(train.sample(1000, random_state=SEED), seed=9))
acc_dikenal = accuracy_score(dikenal["label"], model_eval.predict(dikenal["text"]))
print(f"Akurasi pada produk yang sudah dikenal (noise baru): {acc_dikenal:.1%}")
print("-> Selisih dengan produk baru di atas = 'harga' dari menebak produk yang belum pernah dilihat.")

In [ ]:
test_all = pd.concat([test_bersih, test_noise], ignore_index=True)
pred_all = model_eval.predict(test_all["text"])
print(classification_report(test_all["label"], pred_all, digits=3))

labels = sorted(df["kategori"].unique())
cm = confusion_matrix(test_all["label"], pred_all, labels=labels)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(11, 9))
sns.heatmap(cm_norm, annot=True, fmt=".0%", cmap="Blues", xticklabels=labels, yticklabels=labels, cbar=False)
plt.title("Confusion Matrix (persentase per baris) — data uji"); plt.xlabel("Prediksi"); plt.ylabel("Aktual")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

pasangan = [(labels[i], labels[j], int(cm[i, j])) for i in range(len(labels)) for j in range(len(labels))
            if i != j and cm[i, j] > 0]
print("Kesalahan paling sering (aktual -> prediksi):")
for a, b, n in sorted(pasangan, key=lambda x: -x[2])[:10]:
    print(f"  {a:20} -> {b:20} {n}x")

### Evaluasi lebih ketat: GroupKFold per merek / jenis produk
Pembagian per produk masih membolehkan merek yang sama muncul di training dan test (mis. "Indomie Goreng" di train, "Indomie Soto" di test).
Di sini seluruh produk dari satu `grup` merek disembunyikan sekaligus, jadi model benar-benar menghadapi merek/jenis yang asing.
Angkanya akan lebih rendah, dan itu normal.

In [ ]:
data_semua = pd.concat([train, val, test], ignore_index=True)
akurasi_fold, f1_fold = [], []
for i, (a, b) in enumerate(GroupKFold(n_splits=5).split(data_semua, data_semua["kategori"], data_semua["grup"])):
    tr_f = augment(data_semua.iloc[a], n_noisy=3, seed=100 + i)
    te_f = siapkan(data_semua.iloc[b])
    m = buat_model(**BEST).fit(tr_f["text"], tr_f["label"])
    p = m.predict(te_f["text"])
    akurasi_fold.append(accuracy_score(te_f["label"], p))
    f1_fold.append(f1_score(te_f["label"], p, average="macro"))

print("Akurasi per fold :", np.round(akurasi_fold, 3))
print(f"Rata-rata akurasi: {np.mean(akurasi_fold):.1%} (±{np.std(akurasi_fold):.1%})")
print(f"Rata-rata macro-F1: {np.mean(f1_fold):.3f}")

## 6. Kalibrasi Ambang Keyakinan ("Perlu Review")
Naive Bayes cenderung **terlalu yakin**. Daripada menebak sembarangan, item dengan keyakinan di bawah ambang
diberi status **"Perlu Review"** supaya pengguna memeriksanya di halaman review (sesuai alur proposal).
Ambang dipilih dari data **validasi**, lalu dicek di data uji.

In [ ]:
TARGET_AKURASI = 0.90      # target akurasi untuk item yang diterima otomatis
FALLBACK = "Perlu Review"

def kurva_ambang(model, dset):
    proba = model.predict_proba(dset["text"])
    yakin = proba.max(axis=1)
    tebak = model.classes_[proba.argmax(axis=1)]
    benar = (tebak == dset["label"].values)
    baris = []
    for th in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99, 0.995]:
        diterima = yakin >= th
        baris.append({"ambang": th, "cakupan": diterima.mean(),
                      "akurasi_diterima": benar[diterima].mean() if diterima.any() else np.nan})
    return pd.DataFrame(baris)

kurva_val = kurva_ambang(model_eval, val_all)
display(kurva_val.style.format({"cakupan": "{:.1%}", "akurasi_diterima": "{:.1%}"}))

layak = kurva_val[kurva_val["akurasi_diterima"] >= TARGET_AKURASI]
THRESHOLD = float(layak["ambang"].iloc[0]) if len(layak) else float(kurva_val["ambang"].iloc[-1])
print(f"Ambang terpilih (validasi): {THRESHOLD}")

print("\nCek di data uji:")
kurva_test = kurva_ambang(model_eval, test_all)
display(kurva_test[kurva_test["ambang"] == THRESHOLD].style.format({"cakupan": "{:.1%}", "akurasi_diterima": "{:.1%}"}))

## 7. Training Model Final & Simpan

In [ ]:
# Setelah semua evaluasi selesai, latih ulang dengan SEMUA data (train + val + test)
semua_aug = augment(data_semua, n_noisy=6, seed=3)
model_final = buat_model(**BEST).fit(semua_aug["text"], semua_aug["label"])

print("Sampel training final :", len(semua_aug))
print("Ukuran vocabulary     :", len(model_final.named_steps["tfidf"].vocabulary_))
print("Kelas                 :", [str(c) for c in model_final.classes_])

joblib.dump(model_final, "model_kategori_produk.joblib")
print("Tersimpan -> model_kategori_produk.joblib")

In [ ]:
def prediksi_kategori(nama_produk: str, model=model_final, threshold: float = None, top_k: int = 3):
    threshold = THRESHOLD if threshold is None else threshold
    teks = preprocess(nama_produk)
    if not teks:
        return {"input": nama_produk, "kategori": FALLBACK, "keyakinan": 0.0, "top": []}
    proba = model.predict_proba([teks])[0]
    urut = np.argsort(proba)[::-1][:top_k]
    top = [(str(model.classes_[i]), float(proba[i])) for i in urut]
    kategori = top[0][0] if top[0][1] >= threshold else FALLBACK
    return {"input": nama_produk, "kategori": kategori, "keyakinan": top[0][1], "top": top}

def tabel_prediksi(daftar):
    baris = []
    for n in daftar:
        r = prediksi_kategori(n)
        baris.append({"Input": n, "Kategori": r["kategori"], "Keyakinan": r["keyakinan"],
                      "Tebakan teratas": r["top"][0][0] if r["top"] else "-"})
    return pd.DataFrame(baris).style.format({"Keyakinan": "{:.0%}"})

print("Ambang aktif:", THRESHOLD)

## 8. Demo Berbagai Jenis Struk
Daftar di bawah mencakup struk minimarket, SPBU, restoran, apotek, tagihan/pulsa, dan toko lain. Beberapa nama sengaja **tidak ada**
di dataset untuk melihat perilaku model pada produk baru. Item yang keyakinannya rendah akan berstatus *Perlu Review*.

In [ ]:
demo = [
    # minimarket (singkatan khas struk)
    "INDOMIE GRG SPL 85G", "AQUA BTL 600ML", "SUNLIGHT JRK NIPIS 400ML", "SAMPOERNA MILD 16", "PEPSODENT 190G",
    # SPBU & transportasi
    "PERTAMAX", "PERTALITE", "PARKIR MOTOR", "TIKET KRL COMMUTER LINE", "GOCAR", "TOP UP E-TOLL",
    # restoran / kafe
    "NASI GORENG SPESIAL", "AYAM GEPREK", "ES TEH MANIS", "KOPI SUSU GULA AREN",
    # apotek
    "PANADOL BIRU 10 TAB", "VITAMIN C IPI", "BETADINE 15ML",
    # tagihan
    "TOKEN LISTRIK 50000", "PULSA TELKOMSEL 25000", "TAGIHAN INDIHOME",
    # elektronik, fashion, hiburan, pendidikan, hewan, bayi
    "POWERBANK XIAOMI 10000MAH", "KAOS POLOS HITAM", "TIKET BIOSKOP XXI", "PULPEN PILOT", "WHISKAS POUCH", "PAMPERS M",
    # bukan produk
    "SERVICE CHARGE 5%", "PB1 10%",
    # produk yang TIDAK ada di dataset
    "MIE GACOAN LEVEL 3", "BEBEK GORENG PAK SLAMET", "SOLAR DEX LITE", "ONGKIR SHOPEE", "GRABFOOD ANTAR",
]
tabel_prediksi(demo)

In [ ]:
# Contoh membaca satu struk utuh: ambil baris item -> buang kolom harga -> klasifikasikan
SKIP = re.compile(r"\b(total|subtotal|sub total|tunai|cash|kembali|kembalian|change|bayar|debit|kredit|qris|"
                  r"kasir|terima kasih|thank|tanggal|npwp|telp)\b", re.I)
KOLOM_ANGKA = re.compile(r"(\s+(rp\.?\s*)?[\d.,]+(\s*(l|ltr|x))?){2,}\s*$", re.I)   # minimal 2 kolom angka (qty/harga/jumlah)

def ekstrak_item(teks_struk):
    item = []
    for baris in teks_struk.splitlines():
        b = baris.strip()
        if not b or SKIP.search(b) or not KOLOM_ANGKA.search(b):
            continue
        nama = re.sub(r"(\s+(rp\.?\s*)?[\d.,]+(\s*(l|ltr|x))?)+\s*$", "", b, flags=re.I).strip()
        if len(re.findall(r"[A-Za-z]", nama)) >= 3:
            item.append(nama)
    return item

struk_contoh = """
INDOMARET JL. MERDEKA NO 5
INDOMIE GRG SPL 85G     3   3.500   10.500
AQUA BTL 600ML          2   4.000    8.000
SAMPOERNA MILD 16       1  33.000   33.000
KANTONG PLASTIK         1     200      200
TOTAL                           51.700
TUNAI                          100.000
""".strip()

struk_spbu = """
SPBU PERTAMINA 34.12345
PERTAMAX         5,20 L   Rp 12.400    64.480
TOTAL                                64.480
""".strip()

for nama, teks in [("Minimarket", struk_contoh), ("SPBU", struk_spbu)]:
    print(f"=== Struk {nama}")
    for it in ekstrak_item(teks):
        r = prediksi_kategori(it)
        print(f"  {it:28} -> {r['kategori']:18} ({r['keyakinan']:.0%})")

## 9. Export untuk Flutter (on-device, tanpa API)
Parameter model (vocabulary, IDF, log-probabilitas Naive Bayes) diekspor ke JSON, sehingga aplikasi bisa menghitung kategori sendiri
tanpa Python, server, atau layanan AI eksternal.

**Pipeline yang harus ditiru persis di Dart:**
1. `preprocess()` (lihat bagian 3)
2. *character n-gram* `char_wb`: untuk tiap kata → `" " + kata + " "` → n-gram dari panjang `lo` sampai `hi`
   (kata yang lebih pendek dari n dihitung sekali) — fungsi `char_wb_ngrams` di bawah adalah rujukannya
3. `tf = 1 + ln(jumlah)` → kalikan `idf` → normalisasi L2
4. skor kelas = `class_log_prior + feature_log_prob · vektor` → ambil tertinggi
5. jika probabilitas tertinggi (softmax dari skor) di bawah `threshold` → tampilkan **Perlu Review**

In [ ]:
tfidf = model_final.named_steps["tfidf"]
nb    = model_final.named_steps["nb"]

export = {
    "classes": [str(c) for c in nb.classes_],
    "analyzer": tfidf.analyzer, "ngram_range": list(tfidf.ngram_range),
    "sublinear_tf": tfidf.sublinear_tf, "norm": tfidf.norm, "lowercase": tfidf.lowercase,
    "vocabulary": {k: int(v) for k, v in tfidf.vocabulary_.items()},
    "idf": [round(float(x), 5) for x in tfidf.idf_],
    "class_log_prior": [round(float(x), 6) for x in nb.class_log_prior_],
    "feature_log_prob": [[round(float(x), 4) for x in row] for row in nb.feature_log_prob_],
    "threshold": THRESHOLD, "fallback_label": FALLBACK,
}
with open("model_kategori_produk.json", "w", encoding="utf-8") as f:
    json.dump(export, f, separators=(",", ":"))
print(f"Tersimpan -> model_kategori_produk.json ({Path('model_kategori_produk.json').stat().st_size / 1024 / 1024:.1f} MB)")

def char_wb_ngrams(teks, lo, hi):
    hasil = []
    for kata in teks.split():
        w = " " + kata + " "
        for n in range(lo, hi + 1):
            offset = 0
            hasil.append(w[offset:offset + n])
            while offset + n < len(w):
                offset += 1
                hasil.append(w[offset:offset + n])
            if offset == 0:
                break
    return hasil

def prediksi_manual(nama, ex=export):
    teks = preprocess(nama)
    lo, hi = ex["ngram_range"]
    cnt = {}
    for g in char_wb_ngrams(teks, lo, hi):
        if g in ex["vocabulary"]:
            cnt[g] = cnt.get(g, 0) + 1
    vec = {ex["vocabulary"][g]: (1 + np.log(c)) * ex["idf"][ex["vocabulary"][g]] for g, c in cnt.items()}
    norm = np.sqrt(sum(v * v for v in vec.values()))
    if norm == 0:
        return ex["fallback_label"]
    skor = np.array(ex["class_log_prior"], dtype=float)
    for j, v in vec.items():
        skor += (v / norm) * np.array([row[j] for row in ex["feature_log_prob"]])
    p = np.exp(skor - skor.max()); p /= p.sum()
    return ex["classes"][int(p.argmax())] if p.max() >= ex["threshold"] else ex["fallback_label"]

# Verifikasi: hitungan manual dari JSON harus sama dengan scikit-learn
sampel = df.sample(400, random_state=SEED)["receipt_text"].tolist() + demo
cocok = sum(prediksi_manual(n) == prediksi_kategori(n)["kategori"] for n in sampel)
print(f"Prediksi manual dari JSON = scikit-learn: {cocok}/{len(sampel)} ({cocok / len(sampel):.1%})")

## 10. Uji dengan Struk Asli (sangat disarankan)
Semua angka di atas berasal dari data sintetis. Untuk angka yang bisa dipertanggungjawabkan di laporan:

1. Kumpulkan 30–50 struk asli (minimarket, SPBU, restoran, apotek, dll.), tanpa nama toko/data pribadi.
2. Foto → OCR di aplikasi → salin teks item **apa adanya** (termasuk salah baca).
3. Buat file `struk_asli.csv` dengan dua kolom: `receipt_text,kategori` (kategori harus salah satu dari 15 kategori di atas), minimal 100 baris.
4. Jalankan sel di bawah. **Jangan** gunakan data ini untuk training sebelum evaluasi selesai.

In [ ]:
FILE_ASLI = "struk_asli.csv"
if Path(FILE_ASLI).exists():
    asli = pd.read_csv(FILE_ASLI).dropna(subset=["receipt_text", "kategori"])
    asli["text"] = asli["receipt_text"].map(preprocess)
    asli = asli[asli["text"].str.len() > 0]
    pred = model_final.predict(asli["text"])
    print(f"Jumlah item struk asli: {len(asli)}")
    print(f"Akurasi : {accuracy_score(asli['kategori'], pred):.1%}")
    print(f"Macro-F1: {f1_score(asli['kategori'], pred, average='macro'):.3f}\n")
    print(classification_report(asli["kategori"], pred, digits=3))
    salah = asli.assign(prediksi=pred)[pred != asli["kategori"]][["receipt_text", "kategori", "prediksi"]]
    print("Item yang salah (tambahkan produk serupa ke generator lalu latih ulang):")
    display(salah.head(30))
else:
    print(f"File {FILE_ASLI} belum ada -> lewati. Lihat petunjuk di atas.")

## Ringkasan
- Model: **TF-IDF (character n-gram) + Multinomial Naive Bayes**, dilatih dengan augmentasi noise OCR; 15 kategori.
- Evaluasi memakai **split per produk** (produk uji tidak pernah dilihat) dan **GroupKFold per merek** sebagai uji yang lebih ketat.
- Item dengan keyakinan rendah ditandai **Perlu Review**, bukan ditebak.
- Keluaran: `model_kategori_produk.joblib` (Python) dan `model_kategori_produk.json` (Flutter).
- **Keterbatasan:** dataset sintetis dan terbatas (±1.000 produk). Cakupan "semua struk di Indonesia" tidak bisa dijamin dari data buatan;
  cara memperluasnya: tambahkan produk ke `generate_dataset.py` dan kumpulkan struk asli (bagian 10).